In [3]:
from codeocean.data_asset import DataAssetParams, DataAssetSearchParams,DataAssetAttachParams, Source, AWSS3Source
import pandas as pd
from codeocean.data_asset import DataAssetParams, Source, AWSS3Source
from codeocean import CodeOcean
import os,sys

In [4]:
API_GATEWAY_HOST = "api.allenneuraldynamics.org"
client = CodeOcean(domain="https://codeocean.allenneuraldynamics.org", token=os.getenv("API_SECRET"))
computation_id = os.getenv("CO_COMPUTATION_ID")

In [5]:
ASSET_NAME = 'DRN_5HT_scratch-data'

data_asset_params = DataAssetParams(
    name=ASSET_NAME,
    mount='scratch_data',
    tags=["behavior", "ephys"],
    description=(
        "DRN 5-HT behavior + ephys derived data: the full capsule scratch/ tree "
        "(136,003 files, ~157 GB) synced to "
        "s3://aind-scratch-data/jason_lee/DRN_beh_physiology/ on 2026-09-28. "
        "External storage: this asset references the S3 prefix rather than copying "
        "it, so later changes to that prefix are reflected in the asset. "
        "Mounts read-only at /data/scratch_data -- the writable derived-data tree "
        "is still capsule scratch/."
    ),
    source=Source(
        aws=AWSS3Source(
            bucket="aind-scratch-data",
            prefix="jason_lee/DRN_beh_physiology/",
            public=True,
            keep_on_external_storage=True
        ),
    ),
)

# Guard against duplicates: Code Ocean allows multiple assets with the same name,
# so re-running this cell would silently create a second reference to the same
# 157 GB prefix.
matches = client.data_assets.search_data_assets(
    DataAssetSearchParams(query=ASSET_NAME, limit=50, offset=0)
).results
existing = [a for a in matches if a.name == ASSET_NAME]

if existing:
    print(f"Asset {ASSET_NAME!r} already exists ({', '.join(a.id for a in existing)}); not creating another.")
    data_asset = client.data_assets.get_data_asset(existing[0].id)
else:
    data_asset = client.data_assets.create_data_asset(data_asset_params)
    print(f"Created data asset with ID: {data_asset.id}")

# Creation is async and indexing 136k objects is not instant, so wait for the
# asset to become ready rather than assuming success. No timeout -- interrupt the
# cell if it runs longer than you want.
data_asset = client.data_assets.wait_until_ready(data_asset, polling_interval=15)
print(f"Ready: id={data_asset.id} state={data_asset.state} name={data_asset.name!r}")

Created data asset with ID: db933060-229c-46f6-8d7c-dbf88d48a4d7
Ready: id=db933060-229c-46f6-8d7c-dbf88d48a4d7 state=ready name='DRN_5HT_scratch-data'
